In [1]:
#mising data values
import pandas as pd
import numpy as np

In [5]:
##reading our dataset
data= pd.read_csv('/content/Nassau Candy Distributor(1).csv')

# 1. Validate Date Formats

In [8]:
import pandas as pd

# Convert date columns
data['Order Date'] = pd.to_datetime(data['Order Date'], format='%d-%m-%Y', errors='coerce')
data['Ship Date'] = pd.to_datetime(data['Ship Date'], format='%d-%m-%Y', errors='coerce')

In [9]:
print(data[['Order Date','Ship Date']].isnull().sum())

Order Date    0
Ship Date     0
dtype: int64


# 2. Remove Invalid or Negative Lead Times

In [10]:
data['Lead_Time'] = (data['Ship Date'] - data['Order Date']).dt.days

In [11]:
# View the result:
data[['Order Date','Ship Date','Lead_Time']].head()

,Order Date,Ship Date,Lead_Time
0,2024-01-03,2026-06-30,909
1,2024-01-04,2026-07-01,909
2,2024-01-04,2026-07-01,909
3,2024-01-04,2026-07-01,909
4,2024-01-05,2026-07-05,912


In [12]:
# Find negative lead times:
data[data['Lead_Time'] < 0]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead_Time


In [13]:
# Remove them:
data = data[data['Lead_Time'] >= 0]

# 3. Handle Missing Shipment Records

In [14]:
# Check missing values
data['Ship Date'].isnull().sum()

np.int64(0)

In [15]:
# Remove missing shipment records
data = data.dropna(subset=['Ship Date'])

# 4. Standardize Geographic Fields

In [16]:
## **Your geographic columns are:Country/RegionCity,State/Province,Postal Code,Region
## Clean them like this
geo_columns = ['Country/Region','City','State/Province','Region']

for col in geo_columns:
    data[col] = data[col].str.strip().str.title()

In [17]:
## Check unique state names:
data['State/Province'].unique()

array(['Texas', 'Illinois', 'Pennsylvania', 'Kentucky', 'Georgia',
       'California', 'Virginia', 'Delaware', 'South Carolina', 'Ohio',
       'Louisiana', 'Oregon', 'Arizona', 'Arkansas', 'Michigan',
       'Tennessee', 'Florida', 'Ontario', 'Indiana', 'Nevada',
       'South Dakota', 'New York', 'Wisconsin', 'Washington',
       'New Jersey', 'Missouri', 'North Carolina', 'Colorado', 'Alberta',
       'Utah', 'Minnesota', 'Mississippi', 'Iowa', 'New Mexico',
       'Massachusetts', 'Alabama', 'Idaho', 'Montana', 'Maryland',
       'Connecticut', 'New Hampshire', 'British Columbia', 'Quebec',
       'Nova Scotia', 'Oklahoma', 'Nebraska', 'Maine', 'Kansas',
       'Rhode Island', 'Newfoundland And Labrador', 'New Brunswick',
       'Prince Edward Island', 'District Of Columbia', 'Vermont',
       'Manitoba', 'Saskatchewan', 'Wyoming', 'North Dakota',
       'West Virginia'], dtype=object)

In [18]:
# If you find abbreviations, replace them:
data['State/Province'] = data['State/Province'].replace({
    'Ca':'California',
    'Ny':'New York'
})

# 5. Feature Engineering

In [19]:
# A. Shipping Lead Time
data['Lead_Time'] = (data['Ship Date'] - data['Order Date']).dt.days

In [20]:
# B. Order Month
data['Order_Month'] = data['Order Date'].dt.month_name()

In [21]:
# C. Order Year
data['Order_Year'] = data['Order Date'].dt.year

In [22]:
# D. Ship Month
data['Ship_Month'] = data['Ship Date'].dt.month_name()

In [23]:
# E. Ship Year
data['Ship_Year'] = data['Ship Date'].dt.year

In [24]:
# F. Order Weekday
data['Order_Day'] = data['Order Date'].dt.day_name()

In [25]:
# G. Ship Weekday
data['Ship_Day'] = data['Ship Date'].dt.day_name()

In [26]:
# H. Profit Margin
data['Profit_Margin'] = (data['Gross Profit'] / data['Sales']) * 100

In [27]:
# I. Profit Per Unit
data['Profit_per_Unit'] = data['Gross Profit'] / data['Units']

In [28]:
# J. Cost Per Unit
data['Cost_per_Unit'] = data['Cost'] / data['Units']

# Calculate Shipping Lead Time (days)

In [29]:
# Shipping Lead Time
data['Shipping_Lead_Time'] = (data['Ship Date'] - data['Order Date']).dt.days

# View the new column
data[['Order Date', 'Ship Date', 'Shipping_Lead_Time']].head()

,Order Date,Ship Date,Shipping_Lead_Time
0,2024-01-03,2026-06-30,909
1,2024-01-04,2026-07-01,909
2,2024-01-04,2026-07-01,909
3,2024-01-04,2026-07-01,909
4,2024-01-05,2026-07-05,912


In [30]:
# Check if there are any negative values:
data[data['Shipping_Lead_Time'] < 0]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Order_Month,Order_Year,Ship_Month,Ship_Year,Order_Day,Ship_Day,Profit_Margin,Profit_per_Unit,Cost_per_Unit,Shipping_Lead_Time


In [31]:
# If there are, remove them:
data = data[data['Shipping_Lead_Time'] >= 0]

# 2. Categorize Routes by Factory → Customer Region

In [32]:
data['Division_to_Region'] = data['Division'] + " → " + data['Region']

In [33]:
data[['Division', 'Region', 'Division_to_Region']].head()

,Division,Region,Division_to_Region
0,Chocolate,Interior,Chocolate → Interior
1,Chocolate,Interior,Chocolate → Interior
2,Chocolate,Interior,Chocolate → Interior
3,Chocolate,Interior,Chocolate → Interior
4,Chocolate,Atlantic,Chocolate → Atlantic


# 3. Categorize Routes by Factory → Customer State

In [34]:
data['Division_to_State'] = data['Division'] + " → " + data['State/Province']

In [35]:
data[['Division', 'State/Province', 'Division_to_State']].head()

,Division,State/Province,Division_to_State
0,Chocolate,Texas,Chocolate → Texas
1,Chocolate,Illinois,Chocolate → Illinois
2,Chocolate,Illinois,Chocolate → Illinois
3,Chocolate,Illinois,Chocolate → Illinois
4,Chocolate,Pennsylvania,Chocolate → Pennsylvania


# 4. Group Shipments by Ship Mode

In [36]:
data['Ship Mode'].value_counts()

,count
Ship Mode,
Standard Class,6120
Second Class,1979
First Class,1548
Same Day,547


In [37]:
ship_mode_summary = (
    data.groupby('Ship Mode')
      .size()
      .reset_index(name='Number_of_Shipments')
)

ship_mode_summary

,Ship Mode,Number_of_Shipments
0,First Class,1548
1,Same Day,547
2,Second Class,1979
3,Standard Class,6120


In [38]:
ship_mode_sales = (
    data.groupby('Ship Mode')['Sales']
      .sum()
      .reset_index()
)

ship_mode_sales

,Ship Mode,Sales
0,First Class,21319.39
1,Same Day,7113.67
2,Second Class,27860.22
3,Standard Class,85490.35


In [39]:
ship_mode_summary = (
    data.groupby('Ship Mode')
      .agg({
          'Sales': 'sum',
          'Gross Profit': 'sum',
          'Units': 'sum',
          'Order ID': 'count'
      })
      .rename(columns={'Order ID': 'Total_Shipments'})
      .reset_index()
)

ship_mode_summary

,Ship Mode,Sales,Gross Profit,Units,Total_Shipments
0,First Class,21319.39,14011.09,5724,1548
1,Same Day,7113.67,4700.73,1975,547
2,Second Class,27860.22,18306.52,7546,1979
3,Standard Class,85490.35,56424.46,23409,6120


# Route Definition & Aggregation

In [40]:
data['Route_State'] = data['Division'] + " → " + data['State/Province']

route_state_summary = (
    data.groupby('Route_State')
      .agg(
          Total_Shipments=('Order ID', 'count'),
          Avg_Lead_Time=('Shipping_Lead_Time', 'mean'),
          Lead_Time_Variability=('Shipping_Lead_Time', 'std')
      )
      .reset_index()
)

route_state_summary

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
0,Chocolate → Alabama,56,1286.375000,250.916180
1,Chocolate → Alberta,24,1274.041667,285.159982
2,Chocolate → Arizona,216,1313.935185,262.965796
3,Chocolate → Arkansas,58,1279.844828,285.806572
4,Chocolate → British Columbia,22,1290.318182,308.425091
...,...,...,...,...
111,Sugar → Pennsylvania,3,1396.333333,211.022116
112,Sugar → Rhode Island,1,1273.000000,NaN
113,Sugar → Tennessee,1,1272.000000,NaN
114,Sugar → Texas,4,1456.750000,212.753966


In [41]:
# Option 1: Factory → Customer Region
# Route summary by Factory → Region
route_region_summary = (
    data.groupby(['Division', 'Region'])
      .agg(
          Total_Shipments=('Order ID', 'count'),
          Avg_Lead_Time=('Shipping_Lead_Time', 'mean'),
          Lead_Time_Variability=('Shipping_Lead_Time', 'std')
      )
      .reset_index()
)

route_region_summary

,Division,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
0,Chocolate,Atlantic,2858,1322.277817,255.750748
1,Chocolate,Gulf,1560,1311.908974,263.694306
2,Chocolate,Interior,2271,1324.344782,261.693178
3,Chocolate,Pacific,3155,1322.450713,266.499007
4,Other,Atlantic,109,1323.403670,274.695829
5,Other,Gulf,54,1306.962963,295.306426
6,Other,Interior,55,1266.200000,302.371332
7,Other,Pacific,92,1308.706522,271.033250
8,Sugar,Atlantic,19,1389.263158,273.636588
9,Sugar,Gulf,6,1212.166667,273.436220


In [42]:
# # Option 2: Factory → Customer State
# Route summary by Factory → State
route_state_summary = (
    data.groupby(['Division', 'State/Province'])
      .agg(
          Total_Shipments=('Order ID', 'count'),
          Avg_Lead_Time=('Shipping_Lead_Time', 'mean'),
          Lead_Time_Variability=('Shipping_Lead_Time', 'std')
      )
      .reset_index()
)

route_state_summary

,Division,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
0,Chocolate,Alabama,56,1286.375000,250.916180
1,Chocolate,Alberta,24,1274.041667,285.159982
2,Chocolate,Arizona,216,1313.935185,262.965796
3,Chocolate,Arkansas,58,1279.844828,285.806572
4,Chocolate,British Columbia,22,1290.318182,308.425091
...,...,...,...,...,...
111,Sugar,Pennsylvania,3,1396.333333,211.022116
112,Sugar,Rhode Island,1,1273.000000,NaN
113,Sugar,Tennessee,1,1272.000000,NaN
114,Sugar,Texas,4,1456.750000,212.753966


# 1. Rank Routes from Fastest to Slowest

In [43]:
# Factory → Region
route_region_ranked = route_region_summary.sort_values(
    by='Avg_Lead_Time',
    ascending=True
)

route_region_ranked.head()

,Division,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
9,Sugar,Gulf,6,1212.166667,273.436220
6,Other,Interior,55,1266.200000,302.371332
5,Other,Gulf,54,1306.962963,295.306426
7,Other,Pacific,92,1308.706522,271.033250
1,Chocolate,Gulf,1560,1311.908974,263.694306


In [44]:
# Factory → State
route_state_ranked = route_state_summary.sort_values(
    by='Avg_Lead_Time',
    ascending=True
)

route_state_ranked.head()

,Division,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
83,Other,New Mexico,2,906.0,2.828427
79,Other,Nebraska,1,906.0,NaN
72,Other,Louisiana,3,908.0,1.000000
77,Other,Mississippi,1,908.0,NaN
76,Other,Minnesota,1,909.0,NaN


# 2. Top 10 Most Efficient Routes

In [45]:
# Factory → Region
top10_region = route_region_ranked.head(10)

top10_region

,Division,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
9,Sugar,Gulf,6,1212.166667,273.436220
6,Other,Interior,55,1266.200000,302.371332
5,Other,Gulf,54,1306.962963,295.306426
7,Other,Pacific,92,1308.706522,271.033250
1,Chocolate,Gulf,1560,1311.908974,263.694306
0,Chocolate,Atlantic,2858,1322.277817,255.750748
3,Chocolate,Pacific,3155,1322.450713,266.499007
4,Other,Atlantic,109,1323.403670,274.695829
2,Chocolate,Interior,2271,1324.344782,261.693178
10,Sugar,Interior,9,1354.444444,244.897995


In [46]:
# Factory → State
top10_state = route_state_ranked.head(10)

top10_state

,Division,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
83,Other,New Mexico,2,906.000000,2.828427
79,Other,Nebraska,1,906.000000,NaN
72,Other,Louisiana,3,908.000000,1.000000
77,Other,Mississippi,1,908.000000,NaN
76,Other,Minnesota,1,909.000000,NaN
103,Sugar,Delaware,1,910.000000,NaN
105,Sugar,Illinois,2,1089.000000,257.386868
91,Other,South Carolina,2,1091.000000,257.386868
104,Sugar,Florida,4,1091.500000,210.734430
96,Other,Virginia,9,1109.888889,192.011357


# 3. Bottom 10 Least Efficient Routes

In [47]:
# Factory → Region
bottom10_region = route_region_ranked.tail(10)

bottom10_region

,Division,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
5,Other,Gulf,54,1306.962963,295.306426
7,Other,Pacific,92,1308.706522,271.033250
1,Chocolate,Gulf,1560,1311.908974,263.694306
0,Chocolate,Atlantic,2858,1322.277817,255.750748
3,Chocolate,Pacific,3155,1322.450713,266.499007
4,Other,Atlantic,109,1323.403670,274.695829
2,Chocolate,Interior,2271,1324.344782,261.693178
10,Sugar,Interior,9,1354.444444,244.897995
8,Sugar,Atlantic,19,1389.263158,273.636588
11,Sugar,Pacific,6,1394.500000,188.622109


In [48]:
# Factory → State
bottom10_state = route_state_ranked.tail(10)

bottom10_state

,Division,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
47,Chocolate,Saskatchewan,2,1457.000000,258.801082
33,Chocolate,New Mexico,35,1472.457143,240.290675
85,Other,North Carolina,4,1548.000000,181.356003
109,Sugar,North Carolina,1,1635.000000,NaN
110,Sugar,Ohio,2,1637.500000,3.535534
37,Chocolate,North Dakota,7,1637.857143,1.463850
56,Chocolate,West Virginia,4,1638.000000,2.000000
115,Sugar,Washington,1,1638.000000,NaN
102,Sugar,Connecticut,1,1641.000000,NaN
107,Sugar,New Jersey,1,1642.000000,NaN


In [49]:
# Factory → State
bottom10_state = route_state_ranked.tail(10)

bottom10_state

,Division,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
47,Chocolate,Saskatchewan,2,1457.000000,258.801082
33,Chocolate,New Mexico,35,1472.457143,240.290675
85,Other,North Carolina,4,1548.000000,181.356003
109,Sugar,North Carolina,1,1635.000000,NaN
110,Sugar,Ohio,2,1637.500000,3.535534
37,Chocolate,North Dakota,7,1637.857143,1.463850
56,Chocolate,West Virginia,4,1638.000000,2.000000
115,Sugar,Washington,1,1638.000000,NaN
102,Sugar,Connecticut,1,1641.000000,NaN
107,Sugar,New Jersey,1,1642.000000,NaN


# 4. Compare Performance Across Ship Modes

In [50]:
#Compare each ship mode based on:
# Number of Shipments
# Average Shipping Lead Time
# Lead Time Variability

ship_mode_performance = (
    data.groupby('Ship Mode')
      .agg(
          Total_Shipments=('Order ID', 'count'),
          Avg_Lead_Time=('Shipping_Lead_Time', 'mean'),
          Lead_Time_Variability=('Shipping_Lead_Time', 'std')
      )
      .reset_index()
      .sort_values(by='Avg_Lead_Time')
)

ship_mode_performance

,Ship Mode,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
3,Standard Class,6120,1314.334641,262.400116
2,Second Class,1979,1323.845376,261.813569
1,Same Day,547,1333.442413,253.813374
0,First Class,1548,1338.275840,265.632140


In [51]:
# optional
ship_mode_performance['Rank'] = (
    ship_mode_performance['Avg_Lead_Time']
    .rank(method='dense')
)

ship_mode_performance = ship_mode_performance.sort_values('Rank')

ship_mode_performance

,Ship Mode,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability,Rank
3,Standard Class,6120,1314.334641,262.400116,1.0
2,Second Class,1979,1323.845376,261.813569,2.0
1,Same Day,547,1333.442413,253.813374,3.0
0,First Class,1548,1338.275840,265.632140,4.0


# 1. Analyze Performance by Region

In [52]:
# Calculate:
# Total Shipments
# Average Lead Time
# Lead Time Variability
region_analysis = (
    data.groupby('Region')
      .agg(
          Total_Shipments=('Order ID', 'count'),
          Avg_Lead_Time=('Shipping_Lead_Time', 'mean'),
          Lead_Time_Variability=('Shipping_Lead_Time', 'std')
      )
      .reset_index()
      .sort_values(by='Avg_Lead_Time', ascending=False)
)

region_analysis

,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
2,Interior,2335,1323.091221,262.693597
0,Atlantic,2986,1322.745144,256.541810
3,Pacific,3253,1322.194897,266.470647
1,Gulf,1620,1311.374691,264.727855


# 2. Analyze Performance by State

In [53]:
state_analysis = (
    data.groupby('State/Province')
      .agg(
          Total_Shipments=('Order ID', 'count'),
          Avg_Lead_Time=('Shipping_Lead_Time', 'mean'),
          Lead_Time_Variability=('Shipping_Lead_Time', 'std')
      )
      .reset_index()
      .sort_values(by='Avg_Lead_Time', ascending=False)
)

state_analysis

,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
56,West Virginia,4,1638.000000,2.000000
37,North Dakota,7,1637.857143,1.463850
47,Saskatchewan,2,1457.000000,258.801082
20,Manitoba,12,1455.333333,191.139803
15,Iowa,30,1443.900000,229.801466
33,New Mexico,37,1441.837838,267.198066
53,Vermont,11,1438.909091,190.999714
44,Prince Edward Island,10,1420.300000,255.814190
49,South Dakota,12,1395.916667,360.350955
50,Tennessee,183,1391.486339,248.201971


# 3. Regions with High Shipment Volume + Poor Performance

In [54]:
# A bottleneck often means:
# Many shipments, and High average lead time.
region_bottlenecks = (
    region_analysis
    .sort_values(
        by=['Avg_Lead_Time', 'Total_Shipments'],
        ascending=[False, False]
    )
)

region_bottlenecks

,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
2,Interior,2335,1323.091221,262.693597
0,Atlantic,2986,1322.745144,256.541810
3,Pacific,3253,1322.194897,266.470647
1,Gulf,1620,1311.374691,264.727855


# 4. States with High Shipment Volume + Poor Performance

In [55]:
state_bottlenecks = (
    state_analysis
    .sort_values(
        by=['Avg_Lead_Time', 'Total_Shipments'],
        ascending=[False, False]
    )
)

state_bottlenecks

,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
56,West Virginia,4,1638.000000,2.000000
37,North Dakota,7,1637.857143,1.463850
47,Saskatchewan,2,1457.000000,258.801082
20,Manitoba,12,1455.333333,191.139803
15,Iowa,30,1443.900000,229.801466
33,New Mexico,37,1441.837838,267.198066
53,Vermont,11,1438.909091,190.999714
44,Prince Edward Island,10,1420.300000,255.814190
49,South Dakota,12,1395.916667,360.350955
50,Tennessee,183,1391.486339,248.201971


# 5. Detect Congestion-Prone Regions

In [56]:
congestion_regions = (
    region_analysis
    .sort_values(
        by='Lead_Time_Variability',
        ascending=False
    )
)

congestion_regions.head(10)

,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
3,Pacific,3253,1322.194897,266.470647
1,Gulf,1620,1311.374691,264.727855
2,Interior,2335,1323.091221,262.693597
0,Atlantic,2986,1322.745144,256.541810


# 6. Detect Congestion-Prone States

In [57]:
congestion_states = (
    state_analysis
    .sort_values(
        by='Lead_Time_Variability',
        ascending=False
    )
)

congestion_states.head(10)

,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability
49,South Dakota,12,1395.916667,360.350955
35,Newfoundland And Labrador,6,1216.166667,357.447292
38,Nova Scotia,6,1272.666667,326.914464
45,Quebec,50,1288.240000,313.055421
4,British Columbia,22,1290.318182,308.425091
27,Montana,15,1273.800000,308.316906
18,Louisiana,42,1263.785714,306.777550
24,Minnesota,89,1335.471910,297.651913
1,Alberta,26,1274.076923,292.400673
52,Utah,53,1253.000000,289.780554


# 1. Compare Shipping Efficiency by Ship Mode

In [58]:
# Calculate for each Ship Mode:
# Total Shipments
# Average Shipping Lead Time
# Lead Time Variability
# Total Sales
# Total Cost

ship_mode_analysis = (
    data.groupby('Ship Mode')
      .agg(
          Total_Shipments=('Order ID', 'count'),
          Avg_Lead_Time=('Shipping_Lead_Time', 'mean'),
          Lead_Time_Variability=('Shipping_Lead_Time', 'std'),
          Total_Sales=('Sales', 'sum'),
          Total_Cost=('Cost', 'sum')
      )
      .reset_index()
      .sort_values(by='Avg_Lead_Time')
)

ship_mode_analysis

,Ship Mode,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability,Total_Sales,Total_Cost
3,Standard Class,6120,1314.334641,262.400116,85490.35,29065.89
2,Second Class,1979,1323.845376,261.813569,27860.22,9553.70
1,Same Day,547,1333.442413,253.813374,7113.67,2412.94
0,First Class,1548,1338.275840,265.632140,21319.39,7308.30


# 2. Categorize Ship Modes

In [59]:
# Standard Class → Standard Shipping
# First Class, Second Class, Same Day → Expedited Shipping
data['Shipping_Type'] = data['Ship Mode'].replace({
    'Standard Class': 'Standard Shipping',
    'First Class': 'Expedited Shipping',
    'Second Class': 'Expedited Shipping',
    'Same Day': 'Expedited Shipping'
})

In [60]:
print(data['Ship Mode'].unique())

['Standard Class' 'First Class' 'Second Class' 'Same Day']


# 3. Compare Standard vs Expedited Shipping

In [61]:
shipping_type_analysis = (
    data.groupby('Shipping_Type')
      .agg(
          Total_Shipments=('Order ID', 'count'),
          Avg_Lead_Time=('Shipping_Lead_Time', 'mean'),
          Lead_Time_Variability=('Shipping_Lead_Time', 'std'),
          Total_Sales=('Sales', 'sum'),
          Total_Cost=('Cost', 'sum')
      )
      .reset_index()
)

shipping_type_analysis

,Shipping_Type,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability,Total_Sales,Total_Cost
0,Expedited Shipping,4074,1330.617084,262.240984,56293.28,19274.94
1,Standard Shipping,6120,1314.334641,262.400116,85490.35,29065.89


# 4. Evaluate Cost–Time Tradeoffs

In [62]:
shipping_type_analysis['Avg_Cost_Per_Shipment'] = (
    shipping_type_analysis['Total_Cost'] /
    shipping_type_analysis['Total_Shipments']
)

shipping_type_analysis

,Shipping_Type,Total_Shipments,Avg_Lead_Time,Lead_Time_Variability,Total_Sales,Total_Cost,Avg_Cost_Per_Shipment
0,Expedited Shipping,4074,1330.617084,262.240984,56293.28,19274.94,4.731208
1,Standard Shipping,6120,1314.334641,262.400116,85490.35,29065.89,4.749328
